# L0 vs F1 / MCC curves — hidden_200 runs

Compares all 4 SAE architectures (ReLU, BatchTopK, Matryoshka, Matching Pursuit) trained on the `hidden_200` benchmark setup (`d_in=200`, `d_sae=648`, 60M training samples).

**Main traces** (solid): HuggingFace AE — **Overlay** (faded dotted): SynthAE

Data sources:
- `data/hidden_200_standard_mp/` + `data/hidden_200_standard_mp_synth/` — Standard and Matching Pursuit
- `data/hidden_200_batch_matryoshka/` + `data/hidden_200_batch_matryoshka_synth/` — BatchTopK and Matryoshka (2-level)

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from occhio.benchmark import benchmark_ae_baselines

# ── Main data (HuggingFace AE) ───────────────────────────────────────────────
datasets = {
    "Standard + MP": "data/hidden_200_standard_mp/results.parquet",
    "BatchTopK + Matryoshka": "data/hidden_200_batch_matryoshka/results.parquet",
}

dfs = []
for label, path in datasets.items():
    d = pd.read_parquet(path).reset_index()
    dfs.append(d)

raw = pd.concat(dfs, ignore_index=True)
raw = raw.rename(columns={"f1_score": "f1"})

best = (
    raw.groupby(["benchmark", "sae_type", "sae_l0"], as_index=False)
    .agg(f1=("f1", "max"), mcc=("mcc", "max"))
    .sort_values(["benchmark", "sae_type", "sae_l0"])
    .reset_index(drop=True)
)

# ── Synth data (SynthAE) — shown as semi-transparent overlay ─────────────────
synth_datasets = {
    "Standard + MP (synth)": "data/hidden_200_standard_mp_synth/results.parquet",
    "BatchTopK + Matryoshka (synth)": "data/hidden_200_batch_matryoshka_synth/results.parquet",
}

synth_dfs = []
for label, path in synth_datasets.items():
    d = pd.read_parquet(path).reset_index()
    synth_dfs.append(d)

raw_synth = pd.concat(synth_dfs, ignore_index=True)
raw_synth = raw_synth.rename(columns={"f1_score": "f1"})

best_synth = (
    raw_synth.groupby(["benchmark", "sae_type", "sae_l0"], as_index=False)
    .agg(f1=("f1", "max"), mcc=("mcc", "max"))
    .sort_values(["benchmark", "sae_type", "sae_l0"])
    .reset_index(drop=True)
)

# ── Shared config ─────────────────────────────────────────────────────────────
benchmarks = sorted(best["benchmark"].unique())
arch_order = ["BatchTopK", "Matryoshka", "MatchingPursuit", "Standard"]
colors = {
    "BatchTopK": "#636EFA",
    "Matryoshka": "#EF553B",
    "MatchingPursuit": "#00CC96",
    "Standard": "#AB63FA",
}

# AE baseline F1 per benchmark — computed with SynthAE(n_hidden=200) and
# per-feature threshold sweep on the first 648 features (= d_sae).
ae_baselines = benchmark_ae_baselines(
    device="mps",
    cache_path="data/ae_baselines_hidden_200_per_feature.parquet",
    ae_type="synth",
    ae_kwargs={"n_hidden": 200},
    n_eval_features=648,
)

print(f"Benchmarks: {benchmarks}")
print(f"Architectures: {best['sae_type'].unique().tolist()}")
print(f"Main rows after groupby: {len(best)}, Synth rows: {len(best_synth)}")
print(f"\nAE baselines (hidden_200):\n{ae_baselines}")

In [ ]:
COLOR_AE_BASELINE = "#999999"
SYNTH_OPACITY = 0.35


def _add_arch_traces(
    fig, sub, sub_synth, metric, arch_order, colors, shown_legends, row, col
):
    """Add main + synth overlay traces for all architectures in one subplot."""
    for arch in arch_order:
        color = colors[arch]

        # Synth overlay (drawn first so main sits on top)
        synth_data = sub_synth[sub_synth["sae_type"] == arch].sort_values("sae_l0")
        if not synth_data.empty:
            fig.add_trace(
                go.Scatter(
                    x=synth_data["sae_l0"],
                    y=synth_data[metric],
                    mode="lines+markers",
                    name=arch,
                    line=dict(color=color, dash="dot"),
                    marker=dict(size=4, color=color, opacity=SYNTH_OPACITY),
                    opacity=SYNTH_OPACITY,
                    legendgroup=arch,
                    showlegend=False,
                    hovertemplate=f"{arch} (synth)<br>L0=%{{x}}<br>{metric}=%{{y:.3f}}<extra></extra>",
                ),
                row=row,
                col=col,
            )

        # Main trace
        arch_data = sub[sub["sae_type"] == arch].sort_values("sae_l0")
        if arch_data.empty:
            continue
        show_legend = arch not in shown_legends
        shown_legends.add(arch)
        fig.add_trace(
            go.Scatter(
                x=arch_data["sae_l0"],
                y=arch_data[metric],
                mode="lines+markers",
                name=arch,
                line=dict(color=color),
                marker=dict(size=6, color=color),
                legendgroup=arch,
                showlegend=show_legend,
            ),
            row=row,
            col=col,
        )


def make_l0_plot(
    metric: str, y_label: str, show_ae_baseline: bool = False
) -> go.Figure:
    n_cols = 4
    n_rows = 2
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=benchmarks,
        shared_xaxes=False,
        shared_yaxes=False,
        horizontal_spacing=0.07,
        vertical_spacing=0.14,
    )

    shown_legends = set()
    for idx, benchmark in enumerate(benchmarks):
        row = idx // n_cols + 1
        col = idx % n_cols + 1
        sub = best[best["benchmark"] == benchmark]
        sub_synth = best_synth[best_synth["benchmark"] == benchmark]

        _add_arch_traces(
            fig, sub, sub_synth, metric, arch_order, colors, shown_legends, row, col
        )

        # AE baseline horizontal line
        if show_ae_baseline:
            baseline_key = benchmark.lower()
            if baseline_key in ae_baselines.index:
                baseline_f1 = ae_baselines.loc[baseline_key, "precision_macro"]
                show_bl_legend = "AE baseline" not in shown_legends
                shown_legends.add("AE baseline")
                fig.add_trace(
                    go.Scatter(
                        x=[sub["sae_l0"].min(), sub["sae_l0"].max()],
                        y=[baseline_f1, baseline_f1],
                        mode="lines",
                        name="AE baseline",
                        line=dict(color=COLOR_AE_BASELINE, dash="dash", width=2),
                        legendgroup="AE baseline",
                        showlegend=show_bl_legend,
                    ),
                    row=row,
                    col=col,
                )

        fig.update_xaxes(title_text="L0", row=row, col=col)
        fig.update_yaxes(title_text=y_label if col == 1 else "", row=row, col=col)

    fig.update_layout(
        height=600,
        width=1400,
        title_text=f"Best {y_label} at each L0, per SAE architecture (hidden_200; faded = synth AE)",
        legend=dict(orientation="h", yanchor="bottom", y=1.04, xanchor="right", x=1),
    )
    return fig

In [ ]:
# Graph 1: L0 vs F1 (with AE baseline)
fig_f1 = make_l0_plot("f1", "F1", show_ae_baseline=True)
fig_f1.show()

In [ ]:
# Graph 2: L0 vs MCC
fig_mcc = make_l0_plot("mcc", "MCC")
fig_mcc.show()

In [ ]:
# ── Paper-ready style ────────────────────────────────────────────────────────
PAPER_COLORS = {
    "BatchTopK": "#2166ac",
    "Matryoshka": "#d6604d",
    "MatchingPursuit": "#4daf4a",
    "Standard": "#762a83",
}
PAPER_COLOR_AE = "#000000"
PAPER_SYNTH_OPACITY = 0.35

_FONT = dict(family="Times New Roman, serif", size=28, color="#333333")
_AXIS_STYLE = dict(
    showgrid=False,
    zeroline=False,
    linecolor="#666666",
    linewidth=1,
    ticks="outside",
    ticklen=4,
    tickwidth=1,
    tickcolor="#666666",
    tickfont_size=22,
)
_LINE_WIDTH = 2
_SUBPLOT_TITLE_FONT = dict(family="Times New Roman, serif", size=22, color="#333333")
_AXIS_TITLE_FONT = dict(family="Times New Roman, serif", size=30, color="#333333")

BENCHMARK_LABELS = {
    "CORRELATED_PAIRS": "Correlated Pairs",
    "DAG_RANDOM_WALK": "Deep Hierarchy",
    "HIERARCHICAL_PAIRS": "Hierarchical Pairs",
    "POWER_LAW_DIGRAPH": "Preferential Attachment",
    "SIMPLICIAL_COMPLEX": "Simplicial Complex",
    "SPARSE_UNIFORM": "Sparse Uniform",
    "SPHERICAL": "Spherical",
    "TORUS": "Torus",
}

BENCHMARK_ORDER = [
    "SPARSE_UNIFORM",
    "CORRELATED_PAIRS",
    "HIERARCHICAL_PAIRS",
    "POWER_LAW_DIGRAPH",
    "DAG_RANDOM_WALK",
    "SPHERICAL",
    "SIMPLICIAL_COMPLEX",
    "TORUS",
]

ARCH_LABELS = {
    "BatchTopK": "BatchTopK",
    "Matryoshka": "Matryoshka",
    "MatchingPursuit": "Matching Pursuit",
    "Standard": "ReLU",
}

PAPER_ARCH_ORDER = ["Standard", "BatchTopK", "Matryoshka", "MatchingPursuit"]


def make_l0_f1_paper() -> go.Figure:
    n_cols, n_rows = 4, 2
    subplot_titles = [BENCHMARK_LABELS.get(b, b) for b in BENCHMARK_ORDER]
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=subplot_titles,
        shared_xaxes=False,
        shared_yaxes=False,
        horizontal_spacing=0.07,
        vertical_spacing=0.20,
    )

    x_max = int(best["sae_l0"].max())
    x_padding = 0.3

    shown_legends: set[str] = set()
    for idx, benchmark in enumerate(BENCHMARK_ORDER):
        row = idx // n_cols + 1
        col = idx % n_cols + 1
        sub = best[best["benchmark"] == benchmark]
        sub_synth = best_synth[best_synth["benchmark"] == benchmark]

        for arch in PAPER_ARCH_ORDER:
            color = PAPER_COLORS[arch]
            label = ARCH_LABELS.get(arch, arch)

            # Synth overlay (drawn first so main sits on top)
            synth_data = sub_synth[sub_synth["sae_type"] == arch].sort_values("sae_l0")
            if not synth_data.empty:
                fig.add_trace(
                    go.Scatter(
                        x=synth_data["sae_l0"],
                        y=synth_data["f1"],
                        mode="lines+markers",
                        name=label,
                        line=dict(color=color, width=_LINE_WIDTH, dash="dot"),
                        marker=dict(size=4, color=color, opacity=PAPER_SYNTH_OPACITY),
                        opacity=PAPER_SYNTH_OPACITY,
                        legendgroup=label,
                        showlegend=False,
                        hovertemplate=f"{label} (synth)<br>L0=%{{x}}<br>F1=%{{y:.3f}}<extra></extra>",
                    ),
                    row=row,
                    col=col,
                )

            # Main trace
            arch_data = sub[sub["sae_type"] == arch].sort_values("sae_l0")
            if arch_data.empty:
                continue
            show_legend = label not in shown_legends
            shown_legends.add(label)
            fig.add_trace(
                go.Scatter(
                    x=arch_data["sae_l0"],
                    y=arch_data["f1"],
                    mode="lines+markers",
                    name=label,
                    line=dict(color=color, width=_LINE_WIDTH),
                    marker=dict(size=5, color=color),
                    legendgroup=label,
                    showlegend=show_legend,
                ),
                row=row,
                col=col,
            )

        # AE baseline horizontal dashed line
        baseline_key = benchmark.lower()
        if baseline_key in ae_baselines.index:
            baseline_val = ae_baselines.loc[baseline_key, "precision_macro"]
            show_bl = "AE baseline" not in shown_legends
            shown_legends.add("AE baseline")
            fig.add_trace(
                go.Scatter(
                    x=[1, x_max],
                    y=[baseline_val, baseline_val],
                    mode="lines",
                    name="AE baseline",
                    line=dict(color=PAPER_COLOR_AE, dash="longdash", width=1),
                    legendgroup="AE baseline",
                    showlegend=show_bl,
                ),
                row=row,
                col=col,
            )

        # y=1 reference line spanning full visible x range
        fig.add_trace(
            go.Scatter(
                x=[1 - x_padding, x_max],
                y=[1, 1],
                mode="lines",
                line=dict(color="#000000", width=1, dash="solid"),
                showlegend=False,
                hoverinfo="skip",
            ),
            row=row,
            col=col,
        )

        fig.update_xaxes(
            title_text="L0" if row == n_rows else "",
            title_font=_AXIS_TITLE_FONT,
            range=[1 - x_padding, x_max + x_padding],
            tickmode="linear",
            tick0=1,
            dtick=1,
            minor=dict(ticks=""),
            row=row,
            col=col,
            **_AXIS_STYLE,
        )
        fig.update_yaxes(
            title_text="F1" if col == 1 else "",
            title_font=_AXIS_TITLE_FONT,
            range=[0, 1],
            minor=dict(ticks="outside", ticklen=2),
            row=row,
            col=col,
            **_AXIS_STYLE,
        )

    fig.update_annotations(font=_SUBPLOT_TITLE_FONT)

    fig.update_layout(
        template="plotly_white",
        font=_FONT,
        height=650,
        width=1400,
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=dict(l=60, r=20, t=100, b=50),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.08,
            xanchor="right",
            x=1,
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="#cccccc",
            borderwidth=1,
            font_size=20,
        ),
    )
    return fig


def make_l0_mcc_paper() -> go.Figure:
    n_cols, n_rows = 4, 2
    subplot_titles = [BENCHMARK_LABELS.get(b, b) for b in BENCHMARK_ORDER]
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=subplot_titles,
        shared_xaxes=False,
        shared_yaxes=False,
        horizontal_spacing=0.07,
        vertical_spacing=0.20,
    )

    x_max = int(best["sae_l0"].max())
    x_padding = 0.3

    shown_legends: set[str] = set()
    for idx, benchmark in enumerate(BENCHMARK_ORDER):
        row = idx // n_cols + 1
        col = idx % n_cols + 1
        sub = best[best["benchmark"] == benchmark]
        sub_synth = best_synth[best_synth["benchmark"] == benchmark]

        for arch in PAPER_ARCH_ORDER:
            color = PAPER_COLORS[arch]
            label = ARCH_LABELS.get(arch, arch)

            # Synth overlay (drawn first so main sits on top)
            synth_data = sub_synth[sub_synth["sae_type"] == arch].sort_values("sae_l0")
            if not synth_data.empty:
                fig.add_trace(
                    go.Scatter(
                        x=synth_data["sae_l0"],
                        y=synth_data["mcc"],
                        mode="lines+markers",
                        name=label,
                        line=dict(color=color, width=_LINE_WIDTH, dash="dot"),
                        marker=dict(size=4, color=color, opacity=PAPER_SYNTH_OPACITY),
                        opacity=PAPER_SYNTH_OPACITY,
                        legendgroup=label,
                        showlegend=False,
                        hovertemplate=f"{label} (synth)<br>L0=%{{x}}<br>MCC=%{{y:.3f}}<extra></extra>",
                    ),
                    row=row,
                    col=col,
                )

            # Main trace
            arch_data = sub[sub["sae_type"] == arch].sort_values("sae_l0")
            if arch_data.empty:
                continue
            show_legend = label not in shown_legends
            shown_legends.add(label)
            fig.add_trace(
                go.Scatter(
                    x=arch_data["sae_l0"],
                    y=arch_data["mcc"],
                    mode="lines+markers",
                    name=label,
                    line=dict(color=color, width=_LINE_WIDTH),
                    marker=dict(size=5, color=color),
                    legendgroup=label,
                    showlegend=show_legend,
                ),
                row=row,
                col=col,
            )

        # y=1 reference line spanning full visible x range
        fig.add_trace(
            go.Scatter(
                x=[1 - x_padding, x_max],
                y=[1, 1],
                mode="lines",
                line=dict(color="#000000", width=1, dash="solid"),
                showlegend=False,
                hoverinfo="skip",
            ),
            row=row,
            col=col,
        )

        fig.update_xaxes(
            title_text="L0" if row == n_rows else "",
            title_font=_AXIS_TITLE_FONT,
            range=[1 - x_padding, x_max + x_padding],
            tickmode="linear",
            tick0=1,
            dtick=1,
            minor=dict(ticks=""),
            row=row,
            col=col,
            **_AXIS_STYLE,
        )
        fig.update_yaxes(
            title_text="MCC" if col == 1 else "",
            title_font=_AXIS_TITLE_FONT,
            range=[0, 1],
            minor=dict(ticks="outside", ticklen=2),
            row=row,
            col=col,
            **_AXIS_STYLE,
        )

    fig.update_annotations(font=_SUBPLOT_TITLE_FONT)

    fig.update_layout(
        template="plotly_white",
        font=_FONT,
        height=650,
        width=1400,
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=dict(l=60, r=20, t=100, b=50),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.08,
            xanchor="right",
            x=1,
            bgcolor="rgba(255,255,255,0.9)",
            bordercolor="#cccccc",
            borderwidth=1,
            font_size=20,
        ),
    )
    return fig

In [ ]:
fig_f1_paper = make_l0_f1_paper()
fig_f1_paper.show()

fig_mcc_paper = make_l0_mcc_paper()
fig_mcc_paper.show()

In [ ]:
fig_mcc_paper.write_image("fig_f1_paper_hidden_200_mcc.pdf")